# Étape 3 — MLflow Model Registry & Serving

**Projet MLOps — Home Credit Default Risk**

Ce notebook couvre :
- La connexion au backend SQLite MLflow
- L'enregistrement du meilleur modèle dans le **Model Registry**
- La gestion des versions et **aliases** (champion / challenger)
- Le chargement d'un modèle depuis le registry
- Le **Serving** MLflow via REST API

> ⚠️ **MLflow 3.x** : les *stages* (Staging/Production) sont dépréciés. On utilise désormais des **aliases** (`champion`, `challenger`, etc.)

## 0. Configuration

In [1]:
import mlflow
import mlflow.sklearn
import mlflow.lightgbm
from mlflow import MlflowClient

import pandas as pd
import numpy as np
import requests
import json

# ── Backend SQLite (4 slashes = chemin absolu Linux) ──────────────────────────
MLFLOW_TRACKING_URI = "sqlite:////home/veron/Documents/OpenClassRoom/p6/mlflow.db"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# Nom du modèle dans le registry
MODEL_NAME = "home_credit_scoring"

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

print(f"✅ Tracking URI : {mlflow.get_tracking_uri()}")
print(f"📦 Nom du modèle dans le registry : {MODEL_NAME}")

/home/veron/Documents/OpenClassRoom/p6/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Tracking URI : sqlite:////home/veron/Documents/OpenClassRoom/p6/mlflow.db
📦 Nom du modèle dans le registry : home_credit_scoring


## 1. Identifier le meilleur run

On récupère le run avec la meilleure AUC parmi tous les runs trackés.

In [2]:
# Lister toutes les expériences
experiments = client.search_experiments()
print("Expériences disponibles :")
for exp in experiments:
    print(f"  - [{exp.experiment_id}] {exp.name}")

Expériences disponibles :
  - [2] home-credit-scoring
  - [1] MLflow Demo
  - [0] p6_MLOps


In [3]:
# Chercher le meilleur run sur la métrique auc_val (toutes expériences)
runs = client.search_runs(
    experiment_ids=[exp.experiment_id for exp in experiments],
    filter_string="",
    order_by=["metrics.auc_val DESC"],
    max_results=10
)

# Afficher le classement
print(f"{'Run Name':<30} {'Run ID':<35} {'AUC Val':>10}")
print("-" * 80)
for run in runs:
    name = run.data.tags.get("mlflow.runName", "(sans nom)")
    auc  = run.data.metrics.get("auc_val", float("nan"))
    print(f"{name:<30} {run.info.run_id:<35} {auc:>10.4f}")

# Meilleur run
best_run = runs[0]
best_run_id   = best_run.info.run_id
best_run_name = best_run.data.tags.get("mlflow.runName", "(sans nom)")
best_auc      = best_run.data.metrics.get("auc_val", float("nan"))

print(f"\n🏆 Meilleur run : {best_run_name} — AUC = {best_auc:.4f}")
print(f"   Run ID : {best_run_id}")

Run Name                       Run ID                                 AUC Val
--------------------------------------------------------------------------------
XGBoost                        29209c0b42384b9f8278098c38baac70        0.7537
XGBoost                        9845abb29b4543448b4085fab5ca4845        0.7537
XGBoost                        87e7250043a34cd0bdfc02baba203bd7        0.7537
XGBoost                        c6ba8aefd41a49ab8a259fe3e01c7277        0.7537
XGBoost                        338acde934714bdebb3f5b5593fa40bf        0.7537
LogisticRegression             fb4a263f886e450c9c1e4b42c876f261        0.7144
LogisticRegression             ab0831484d6f4d90ae81b89d60b1532d        0.7144
LogisticRegression             34f23a766687439ea34f059223220264        0.7144
LogisticRegression             205e545bd7f3492592802859191e3210        0.7144
luminous-bird-406              1457678f49674e9686c723a475312b8c        0.6335

🏆 Meilleur run : XGBoost — AUC = 0.7537
   Run ID : 29209c0b

## 2. Enregistrement dans le Model Registry

On enregistre le modèle du meilleur run dans le registry.

> 💡 L'artifact path dépend du type de modèle loggué (sklearn → `model`, lightgbm → `model`, etc.).
> On détecte automatiquement l'artifact disponible.

In [4]:
# Lister les artifacts du meilleur run pour trouver le bon chemin
artifacts = client.list_artifacts(best_run_id)
print(f"Artifacts du run '{best_run_name}' :")
for a in artifacts:
    print(f"  {'[DIR]' if a.is_dir else '[FILE]'} {a.path}")

# Détection : uniquement les DOSSIERS (un modèle est toujours un dossier)
dirs = [a.path for a in artifacts if a.is_dir]
print(f"\nDossiers trouvés : {dirs}")

Artifacts du run 'XGBoost' :
  [FILE] feature_importance_weight.json
  [FILE] feature_importance_weight.png

Dossiers trouvés : []


In [5]:
# MLflow 3.x — le modèle est déjà enregistré via registered_model_name
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
latest = sorted(versions, key=lambda v: int(v.version), reverse=True)[0]
version = latest.version

print(f"✅ Version trouvée dans le registry : {version}")
print(f"   Run source : {latest.run_id[:8]}...")

✅ Version trouvée dans le registry : 2
   Run source : 29209c0b...


In [6]:
# Cellule remplacée — dans MLflow 3.x, le modèle est déjà dans le registry
# Rien à faire ici, on récupère juste la version existante
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
latest = sorted(versions, key=lambda v: int(v.version), reverse=True)[0]
version = latest.version

print(f"✅ Version dans le registry : {version}")
print(f"   Run source : {latest.run_id[:8]}...")

✅ Version dans le registry : 2
   Run source : 29209c0b...


## 3. Ajouter une description et des tags

Bonne pratique MLOps : annoter chaque version avec ses métriques et son contexte.

In [7]:
# Description de la version
client.update_model_version(
    name=MODEL_NAME,
    version=version,
    description=(
        f"Modèle de scoring crédit — Home Credit Default Risk.\n"
        f"Run source : {best_run_name} (ID: {best_run_id}).\n"
        f"AUC validation : {best_auc:.4f}.\n"
        f"Dataset : application_train.csv uniquement."
    )
)

# Tags sur la version
client.set_model_version_tag(MODEL_NAME, version, "auc_val", f"{best_auc:.4f}")
client.set_model_version_tag(MODEL_NAME, version, "run_name", best_run_name)
client.set_model_version_tag(MODEL_NAME, version, "dataset", "application_train.csv")
client.set_model_version_tag(MODEL_NAME, version, "class_imbalance", "handled")

# Description du modèle (globale)
client.update_registered_model(
    name=MODEL_NAME,
    description="Modèle de scoring de risque de crédit — Projet MLOps OpenClassrooms P6."
)

print("✅ Description et tags ajoutés.")

✅ Description et tags ajoutés.


## 4. Gestion des aliases (MLflow 3.x)

En MLflow 3.x, on remplace les *stages* par des **aliases** :

| Ancienne approche (dépréciée) | Nouvelle approche MLflow 3.x |
|---|---|
| `stage="Staging"` | `alias="challenger"` |
| `stage="Production"` | `alias="champion"` |
| `stage="Archived"` | (supprimer l'alias) |

In [8]:
# ── Assigner l'alias 'champion' à la version courante ─────────────────────────
client.set_registered_model_alias(
    name=MODEL_NAME,
    alias="champion",
    version=version
)

print(f"✅ Alias 'champion' assigné à la version {version} de '{MODEL_NAME}'")

# Si tu veux aussi un alias 'challenger' sur une version précédente :
# client.set_registered_model_alias(MODEL_NAME, "challenger", version="1")

✅ Alias 'champion' assigné à la version 2 de 'home_credit_scoring'


In [9]:
# ── Vérifier l'état du registry ───────────────────────────────────────────────
model_details = client.get_registered_model(MODEL_NAME)
print(f"Modèle : {model_details.name}")
print(f"Description : {model_details.description}")
print(f"\nVersions disponibles :")

versions = client.search_model_versions(f"name='{MODEL_NAME}'")
for v in versions:
    aliases_str = ", ".join(v.aliases) if v.aliases else "(aucun)"
    print(f"  Version {v.version} | Aliases: {aliases_str} | Source run: {v.run_id[:8]}...")
    print(f"    AUC: {v.tags.get('auc_val', 'N/A')} | Run: {v.tags.get('run_name', 'N/A')}")

Modèle : home_credit_scoring
Description : Modèle de scoring de risque de crédit — Projet MLOps OpenClassrooms P6.

Versions disponibles :
  Version 2 | Aliases: (aucun) | Source run: 29209c0b...
    AUC: 0.7537 | Run: XGBoost
  Version 1 | Aliases: (aucun) | Source run: 9845abb2...
    AUC: N/A | Run: N/A


## 5. Charger et utiliser un modèle depuis le registry

Deux façons de charger :
- Par **alias** (recommandé en prod) : `models:/home_credit_scoring@champion`
- Par **version** (déterministe) : `models:/home_credit_scoring/2`

In [10]:
# ── Chargement par alias ──────────────────────────────────────────────────────
champion_uri = f"models:/{MODEL_NAME}@champion"
print(f"Chargement depuis : {champion_uri}")

champion_model = mlflow.pyfunc.load_model(champion_uri)
print(f"✅ Modèle champion chargé !")
print(f"   Type de flaveur : {type(champion_model._model_impl).__name__}")

Chargement depuis : models:/home_credit_scoring@champion
✅ Modèle champion chargé !
   Type de flaveur : _XGBModelWrapper


In [11]:
DATA_PATH = '/home/veron/Documents/OpenClassRoom/p6/dataset/'
app_train = pd.read_csv(DATA_PATH + 'application_train.csv')

# Supprimer uniquement TARGET, garder SK_ID_CURR comme pendant l'entraînement
sample = app_train.drop(columns=['TARGET'], errors='ignore').head(5)

from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

sample_enc = sample.copy()
le = LabelEncoder()
for col in sample_enc.select_dtypes(include='object').columns:
    sample_enc[col] = le.fit_transform(sample_enc[col].astype(str))

imputer = SimpleImputer(strategy='median')
sample_imputed = pd.DataFrame(
    imputer.fit_transform(sample_enc),
    columns=sample_enc.columns
)

predictions = champion_model.predict(sample_imputed)
print("Probabilités de défaut pour les 5 premiers clients :")
for i, pred in enumerate(predictions):
    risk = "⚠️ RISQUÉ" if pred > 0.5 else "✅ OK"
    print(f"  Client {i+1} : P(défaut) = {pred:.4f}  {risk}")

Probabilités de défaut pour les 5 premiers clients :
  Client 1 : P(défaut) = 0.0000  ✅ OK
  Client 2 : P(défaut) = 0.0000  ✅ OK
  Client 3 : P(défaut) = 0.0000  ✅ OK
  Client 4 : P(défaut) = 0.0000  ✅ OK
  Client 5 : P(défaut) = 0.0000  ✅ OK


## 6. MLflow Serving — REST API

### 6.1 Lancer le serveur (dans un terminal)

MLflow peut exposer le modèle comme une **API REST** en une commande.

```bash
# Activer l'environnement virtuel
source ~/Documents/OpenClassRoom/p6/.venv/bin/activate

# Lancer le serving sur le port 5001
mlflow models serve \
  --model-uri "models:/home_credit_scoring@champion" \
  --backend-store-uri sqlite:////home/veron/Documents/OpenClassRoom/p6/mlflow.db \
  --port 5001 \
  --no-conda
```

> 💡 `--no-conda` évite la recréation d'un environnement Conda et utilise le venv Python courant.  
> Le serveur écoute sur `http://127.0.0.1:5001`

### 6.2 Tester l'API REST depuis Python

Une fois le serveur lancé dans un terminal (**cellule ci-dessous à exécuter après avoir lancé le serveur**) :

In [15]:
# ── Test de l'API REST (serveur doit être démarré dans un terminal) ────────────
import requests, json

SERVING_URL = "http://127.0.0.1:5001/invocations"

# Préparer les données au format attendu par MLflow Serving
payload = {
    "dataframe_split": {
        "columns": list(sample_imputed.columns),
        "data": sample_imputed.values.tolist()
    }
}

try:
    response = requests.post(
        SERVING_URL,
        headers={"Content-Type": "application/json"},
        data=json.dumps(payload),
        timeout=10
    )

    if response.status_code == 200:
        result = response.json()
        print("✅ Réponse du serveur MLflow :")
        predictions_api = result.get("predictions", result)
        for i, pred in enumerate(predictions_api):
            p = float(pred)  # ← pred est directement un int/float
            risk = "⚠️ RISQUÉ" if p > 0.5 else "✅ OK"
            print(f"  Client {i+1} : P(défaut) = {p:.0f}  {risk}")
    else:
        print(f"❌ Erreur HTTP {response.status_code} : {response.text}")

except requests.exceptions.ConnectionError:
    print("⚠️  Serveur non démarré. Lance d'abord 'mlflow models serve ...' dans un terminal.")
    print("   Voir la cellule 6.1 ci-dessus pour la commande exacte.")

✅ Réponse du serveur MLflow :
  Client 1 : P(défaut) = 0  ✅ OK
  Client 2 : P(défaut) = 0  ✅ OK
  Client 3 : P(défaut) = 0  ✅ OK
  Client 4 : P(défaut) = 0  ✅ OK
  Client 5 : P(défaut) = 0  ✅ OK


### 6.3 Tester l'API avec curl (dans un terminal)

Alternative depuis le terminal :

```bash
curl -X POST http://127.0.0.1:5001/invocations \
  -H 'Content-Type: application/json' \
  -d '{
    "dataframe_split": {
      "columns": ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY"],
      "data": [[135000.0, 312682.5, 29686.5]]
    }
  }'
```

> ℹ️ Passe les colonnes réelles de ton modèle à la place des exemples ci-dessus.

## 7. Scénario de mise à jour : enregistrer un nouveau challenger

Workflow pour tester un nouveau modèle sans remplacer le champion :

In [16]:
# ── Workflow de promotion champion / challenger ────────────────────────────────
# Exemple : supposons qu'on vient d'entraîner un nouveau run (run_id_nouveau)
# et qu'on veut le tester avant de le promouvoir

def register_as_challenger(run_id: str, artifact_path: str = "model") -> str:
    """Enregistre un run comme nouvelle version 'challenger'."""
    model_uri = f"runs:/{run_id}/{artifact_path}"
    
    registered = mlflow.register_model(model_uri=model_uri, name=MODEL_NAME)
    new_version = registered.version
    
    # Assigner l'alias challenger
    client.set_registered_model_alias(MODEL_NAME, "challenger", new_version)
    
    auc = client.get_run(run_id).data.metrics.get("auc_val", float("nan"))
    client.set_model_version_tag(MODEL_NAME, new_version, "auc_val", f"{auc:.4f}")
    
    print(f"✅ Version {new_version} enregistrée comme 'challenger' (AUC={auc:.4f})")
    return new_version


def promote_challenger_to_champion():
    """Promeut le challenger en champion."""
    challenger_mv = client.get_model_version_by_alias(MODEL_NAME, "challenger")
    champion_mv   = client.get_model_version_by_alias(MODEL_NAME, "champion")
    
    print(f"Champion actuel   : version {champion_mv.version} (AUC={champion_mv.tags.get('auc_val', 'N/A')})")
    print(f"Challenger à promouvoir : version {challenger_mv.version} (AUC={challenger_mv.tags.get('auc_val', 'N/A')})")
    
    # Archiver l'ancien champion (supprimer son alias champion)
    client.delete_registered_model_alias(MODEL_NAME, "champion")
    
    # Promouvoir le challenger
    client.set_registered_model_alias(MODEL_NAME, "champion", challenger_mv.version)
    client.delete_registered_model_alias(MODEL_NAME, "challenger")
    
    print(f"✅ Version {challenger_mv.version} est maintenant le 'champion' !")


# Exemple d'utilisation :
# new_v = register_as_challenger(run_id="abc123", artifact_path="model")
# promote_challenger_to_champion()
print("Fonctions register_as_challenger() et promote_challenger_to_champion() définies.")

Fonctions register_as_challenger() et promote_challenger_to_champion() définies.


## 8. Récapitulatif — Commandes utiles

| Action | Commande / Code |
|---|---|
| Lancer l'UI MLflow | `mlflow ui --backend-store-uri sqlite:////home/veron/Documents/OpenClassRoom/p6/mlflow.db --port 5000` |
| Lancer le serving | `mlflow models serve --model-uri models:/home_credit_scoring@champion --backend-store-uri sqlite:////home/veron/.../mlflow.db --port 5001 --no-conda` |
| Charger le champion | `mlflow.pyfunc.load_model("models:/home_credit_scoring@champion")` |
| Charger une version | `mlflow.pyfunc.load_model("models:/home_credit_scoring/2")` |
| Tester l'API | `curl -X POST http://127.0.0.1:5001/invocations -H 'Content-Type: application/json' -d '{...}'` |

In [17]:
# ── Checklist finale ───────────────────────────────────────────────────────────
checklist = [
    ("✅", "MLflow tracking URI → SQLite"),
    ("✅", "Meilleur run identifié par AUC"),
    ("✅", "Modèle enregistré dans le Model Registry"),
    ("✅", "Description et tags ajoutés à la version"),
    ("✅", "Alias 'champion' assigné (MLflow 3.x)"),
    ("✅", "Modèle chargé depuis le registry via alias"),
    ("⬜", "Serveur de serving démarré (mlflow models serve)"),
    ("⬜", "API REST testée avec requête JSON"),
]

print("\n📋 Checklist MLOps — Registry & Serving")
print("=" * 50)
for status, item in checklist:
    print(f"  {status} {item}")


📋 Checklist MLOps — Registry & Serving
  ✅ MLflow tracking URI → SQLite
  ✅ Meilleur run identifié par AUC
  ✅ Modèle enregistré dans le Model Registry
  ✅ Description et tags ajoutés à la version
  ✅ Alias 'champion' assigné (MLflow 3.x)
  ✅ Modèle chargé depuis le registry via alias
  ⬜ Serveur de serving démarré (mlflow models serve)
  ⬜ API REST testée avec requête JSON


In [18]:
import requests, json
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

DATA_PATH = '/home/veron/Documents/OpenClassRoom/p6/dataset/'
app_train = pd.read_csv(DATA_PATH + 'application_train.csv')
sample = app_train.drop(columns=['TARGET'], errors='ignore').head(3)

le = LabelEncoder()
for col in sample.select_dtypes(include='object').columns:
    sample[col] = le.fit_transform(sample[col].astype(str))

imputer = SimpleImputer(strategy='median')
sample_imputed = pd.DataFrame(imputer.fit_transform(sample), columns=sample.columns)

payload = {
    "dataframe_split": {
        "columns": sample_imputed.columns.tolist(),
        "data": sample_imputed.values.tolist()
    }
}

response = requests.post(
    "http://127.0.0.1:5001/invocations",
    headers={"Content-Type": "application/json"},
    data=json.dumps(payload)
)
print(response.json())

{'predictions': [0, 0, 0]}
